# Diffusion4Distribution —— 算法原理走查（可运行）

**目标**：用一个完全可运行的 notebook，从原理讲清这篇论文的核心算法。

**一句话概括**：给定一个总体，按已知速率 $p$ 抽样得到样本（每个元素独立以概率 $p$ 保留）。论文把这个过程建模为 **加噪（diffusion 的 forward）**，把"从样本恢复总体频数直方图"建模为 **去噪（reverse / denoising）**。核心贡献是：**在 Gamma–Poisson 共轭下，反向一步的精确后验分布有闭式解**——不需要训练任何网络就能恢复每个类型的真实频数，并给出校准的不确定性带。

本 notebook 的所有数值都由 `d4d_logic.py` 实际算出（不靠记忆填数）。它覆盖：
1. 前向过程 = 二项稀疏（binomial thinning）
2. 闭式解析逆 = Gamma–Poisson 共轭后验
3. 整条频谱的重建与 W1 误差
4. 90% plausible set（不确定性带）
5. 学习组件（Thickener / endpoint regression）的定位与诚实局限
6. 三条方法边界
7. 代码 → 论文 的对照表

> 运行环境：`experiments/code/` 下的 `py311` 环境（已装 numpy / scipy / matplotlib）。把本 notebook 放在 `experiments/code/` 目录运行即可，`d4d_logic` 会被自动加入路径。


In [ ]:
import sys, math
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))   # 让本 notebook 能 import d4d_logic

import numpy as np
from scipy.special import gammaln
import matplotlib.pyplot as plt

from d4d_logic import (zipf_population, thin, fofo_spectrum, analytic_posterior,
                      plausible_set, w1_distance, Bucketizer, build_empirical_prior)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (7.2, 3.6)
rng = np.random.default_rng(20261106)
print('imports OK')

## 1. 前向过程：采样 = 二项稀疏（加噪）

数据库里的一个列（总体）是一组类型计数 $c = (c_1, \dots, c_V)$，其中 $c_v$ 是类型 $v$ 在总体里出现的次数。$V$ 是类型字典大小。

按采样率 $p$ 抽样，等价于 **每个元素独立地以概率 $p$ 被保留、以概率 $1-p$ 被丢弃**。这就是 `thin()`：对类型 $v$，观测计数 $g_v \sim \mathrm{Bin}(c_v, p)$。

两个后果：
- **缺失质量（C1）**：真实出现 1 次的类型，以概率 $1-p$ 在样本里完全消失（变成"未见类型"）。
- **假降级（C2）**：真实计数 $c_v$ 被压低成更小的 $g_v$。

下面的单元生成一个 Zipf 总体并抽样，画出前若干类型的真实 vs 观测计数。

In [ ]:
alpha, n, vocab, p = 1.5, 12_000, 2_000, 0.1
c_true = zipf_population(rng, n, vocab, alpha)
c_obs = thin(c_true, p, rng)

print('总体字典大小 V =', int((c_true > 0).sum()))
print('样本中出现的类型数 =', int((c_obs > 0).sum()))
print('消失（未见）的类型数 =', int(((c_true > 0) & (c_obs == 0)).sum()))

top = 12
idx = np.argsort(-c_true)[:top]
fig, ax = plt.subplots(1, 2, figsize=(8.4, 3.4))
ax[0].bar(np.arange(top), c_true[idx], label='true c_v')
ax[0].bar(np.arange(top), c_obs[idx], alpha=0.55, label='observed g_v')
ax[0].set_xlabel('top types'); ax[0].set_ylabel('count'); ax[0].legend(); ax[0].set_title('counts of top-12 types')
ax[1].scatter(c_true[idx], c_obs[idx], s=18)
ax[1].plot([0, c_true[idx].max()], [0, c_true[idx].max()*p], 'k--', lw=1, label='y = p x')
ax[1].set_xlabel('true c_v'); ax[1].set_ylabel('observed g_v'); ax[1].legend(); ax[1].set_title('g_v vs c_v (p=0.1)')
plt.tight_layout(); plt.show()

注意：纵使是计数很高的类型，$g_v$ 也只是 $c_v$ 的 $p$ 倍左右——这正是"已知采样率"的含义。未见类型（C1）不会出现在任何频谱里，所以频谱本身（frequency-of-frequencies，$F_j = $ 出现恰好 $j$ 次的类型数）只包含"在样本里至少出现一次"的类型。

In [ ]:
# 频率频谱 F_j：出现恰好 j 次的类型数（j>=1，不含 F_0）
spec_obs = fofo_spectrum(c_obs)
spec_true_obs = fofo_spectrum(c_true[c_obs >= 1])   # 用"在样本里出现过的类型"的真实计数构造真值频谱
print('F_1 (singletons) observed =', int(spec_obs[0]), '| true among observed types =', int(spec_true_obs[0]))
print('F_2 observed =', int(spec_obs[1]), '| true =', int(spec_true_obs[1]))

## 2. 闭式解析逆：Gamma–Poisson 共轭后验

对每个**观测类型**（即 $g_v \ge 1$ 的类型），我们要从 $g_v$ 反推 $c_v$。这是一个经典的逆抽样问题：

$$P(c \mid g) \propto P_{\text{prior}}(c) \cdot \mathrm{Bin}(g \mid c, p), \qquad c \ge g.$$

论文的关键点：当先验取 **Gamma–Poisson**（对数线性偏置的计数分布族）时，这个后验有**精确闭式**。

设先验为 $\lambda \sim \mathrm{Gamma}(a_0, b_0)$，则 $c \mid \lambda \sim \mathrm{Poisson}(\lambda)$，边缘先验仍是 Gamma–Poisson。在固定 $g$ 下，后验均值与方差是仿射/二次的：

$$\mathbb{E}[c \mid g] = g + (1-p)\,\frac{a_0 + g}{b_0 + p}$$

$$\mathrm{Var}[c \mid g] = (1-p)\,\frac{a_0 + g}{b_0 + p} \;+\; (1-p)^2\,\frac{a_0 + g}{(b_0 + p)^2}$$

下面先用 `analytic_posterior`（代码里的**通用离散枚举**版本，对任意先验都适用）构造 Gamma–Poisson 先验，再核对它的均值/方差与上面的闭式逐位一致。

In [ ]:
def gamma_poisson_prior(a0, b0, N):
    # Build Gamma-Poisson marginal P(c), c = 0..N, then return (prior, logfact).
    fs = np.arange(N + 1)
    logP = (gammaln(a0 + fs) - gammaln(a0) - gammaln(fs + 1)
            + a0 * math.log(b0 / (b0 + 1)) + fs * math.log(1 / (b0 + 1)))
    prior = np.exp(logP); prior = prior / prior.sum()
    logfact = np.concatenate([[0.0], np.cumsum(np.log(np.arange(1, N + 513, dtype=float)))])
    return prior, logfact

a0, b0, N = 0.25, 0.0170164363, 6000
prior, logfact = gamma_poisson_prior(a0, b0, N)

print(f"{'g':>4} | {'enum mean':>11} {'closed mean':>11} | {'enum var':>11} {'closed var':>11}")
for g in [1, 5, 20, 100]:
    fsup, prob = analytic_posterior(g, p, prior, logfact, N)
    em = float((fsup * prob).sum())
    ev = float(((fsup - em) ** 2 * prob).sum())
    c = (a0 + g) / (b0 + p)
    cm = g + (1 - p) * c
    cv = (1 - p) * c + (1 - p) ** 2 * c / (b0 + p)
    print(f"{g:>4} | {em:>11.5f} {cm:>11.5f} | {ev:>11.5f} {cv:>11.5f}")

枚举（代码）与闭式（公式）在机器精度上完全一致——这证明了 `analytic_posterior` 在 Gamma–Poisson 下确实退化为那条仿射后验均值。注意修正项 $(1-p)\frac{a_0+g}{b_0+p}$：它同时补偿了 C1（缺失质量）和 C2（假降级），这正是这个算法与 Good–Turing / 经典无偏估计器的本质区别——我们**显式建模了 Bernoulli 信道并给出有限总体的后验**，而不是在 iid 生成分布下给一个修正标量。

下面把**完整后验分布** $P(c\mid g)$ 画出来（取一个具体 $g$），并叠加闭式均值位置。

In [ ]:
g_demo = 20
fsup, prob = analytic_posterior(g_demo, p, prior, logfact, N)
c = (a0 + g_demo) / (b0 + p)
closed_mean = g_demo + (1 - p) * c

fig, ax = plt.subplots()
ax.stem(fsup, prob, basefmt=' ')
ax.axvline(closed_mean, color='red', ls='--', lw=1, label=f'closed mean = {closed_mean:.2f}')
ax.axvline(g_demo / p, color='green', ls=':', lw=1, label=f'naive g/p = {g_demo/p:.2f}')
ax.set_xlim(g_demo, g_demo + 60)
ax.set_xlabel('true count c'); ax.set_ylabel('P(c | g)')
ax.set_title(f'posterior of c given observed g = {g_demo} (p = 0.1)')
ax.legend(); plt.tight_layout(); plt.show()

绿虚线是朴素估计 $g/p$，红虚线是闭式后验均值。两者之差就是缺失质量 + 假降级的修正量。

## 3. 逐类型恢复：后验均值 vs 朴素 $g/p$

论文的头号交付物是**每个观测类型的后验**，而不是一条频谱。先看最简单也最诚实的检验：对样本里出现过的每个类型，用后验均值 $\hat c_v = \mathbb{E}[c_v\mid g_v]$ 反推，和朴素估计 $g_v/p$ 比，谁更接近真实计数 $c_v$。

这里比较的是**逐类型**恢复精度（论文 Table II 的动机）。频谱级的 W1 会被长尾主导，而点估计在尾部会过度修正——论文明确写了"重尾精度仍归于原始样本"，所以**不要拿 W1 来宣称逆变换 uniformly 优于原始样本**；它的价值在于逐类型后验与下面第 4 节的校准带。

**一个诚实提醒**：朴素估计 $g/p$ 对 $c$ 是无偏的（因为 $\mathbb E[g]=pc$）。后验均值只有在**先验与总体同形状、同规模**时才稳赢无偏估计；一旦先验错配（支持边界或形状不符），后验均值会被拉向先验均值、偏差变大，点估计反而退化到接近朴素——这正是论文边界 #2（解析逆不 uniform 支配最强线性估计器）在逐类型层面的体现。下面用的是与总体同形状同规模的"预言经验先验"（论文展示校准的最佳情形）。

In [ ]:
# 用与真实总体同形状、同规模的"预言经验先验"（论文用来展示校准的最佳情形）。
# 关键：先验的支持边界 N 必须与总体规模一致；否则后验均值会被错误拉偏。
N = 12_000
prior, logfact = build_empirical_prior(alpha, N, vocab, n_pops=24, seed=20_000, rng=rng)

def posterior_mean_per_type(c_obs, prior, logfact, N, p):
    g_obs = c_obs.astype(np.int64)
    rec = np.zeros_like(g_obs, dtype=float)
    for gi in np.where(g_obs >= 1)[0]:
        fs, pr = analytic_posterior(int(g_obs[gi]), p, prior, logfact, N)
        rec[gi] = float((fs * pr).sum())
    return rec

rec_mean = posterior_mean_per_type(c_obs, prior, logfact, N, p)
obs_idx = np.where(c_obs >= 1)[0]
true_vals = c_true[obs_idx].astype(float)
g_vals = c_obs[obs_idx].astype(float)
naive = g_vals / p
mae_inv = np.mean(np.abs(rec_mean[obs_idx] - true_vals))
mae_naive = np.mean(np.abs(naive - true_vals))
print(f'per-type MAE  naive g/p      = {mae_naive:.3f}')
print(f'per-type MAE  posterior mean = {mae_inv:.3f}')
print(f'improvement over naive       = {100 * (1 - mae_inv / mae_naive):.1f}%')

print('\\n   g   true   naive(g/p)   posterior-mean')
for k in [0, 3, 10, 30, 80, 130]:
    if k < len(obs_idx):
        j = int(obs_idx[k])
        print(f'  {int(c_obs[j]):4d}  {int(c_true[j]):5d}   {c_obs[j] / p:10.1f}   {rec_mean[j]:10.1f}')

# illustrative spectrum shape (NOT a W1 win claim)
j_plot = 60
fig, ax = plt.subplots()
ax.plot(fofo_spectrum(c_true)[:j_plot], 'k-', lw=2, label='true spectrum')
ax.plot(fofo_spectrum(c_obs)[:j_plot], 'gray', label='raw observed')
ax.plot(fofo_spectrum(rec_mean)[:j_plot], 'C0', label='posterior-mean inverse')
ax.set_yscale('log'); ax.set_xlabel('j (exact frequency)'); ax.set_ylabel('F_j (log)')
ax.set_title('shape recovery (illustrative; tail is biased)')
ax.legend(); plt.tight_layout(); plt.show()

后验均值把每个观测类型的计数向上修正（补偿 C1 缺失质量与 C2 降级），逐类型 MAE 比朴素 $g/p$ 更低。但尾部仍会过度修正——这正是论文诚实边界 #1：重尾精度仍归于原始样本。本 notebook 的演示用 **offline profiling** 口径，论文不主张把它接入实时优化器或网络监控。

## 4. 复原效果：FoF 折线图与完整总体真值对比

这一节直接回答复原出来的频谱长什么样、跟真值差多少。

**真值的定义**。$F_j$ 是完整总体中真实频次恰好等于 $j$ 的 distinct value 数量。图上每条曲线都是对同一个 $F_j$ 的估计，共享同一个 $j$ 轴与同一个度量空间。纵轴是绝对类型数，取 log 轴。

**为什么不放观测样本的 FoF**。观测样本的 FoF 统计的是样本频次等于 $j$ 的类型数，它的轴是稀疏后的 $g$ 而不是真实频次 $f$，支持集也只有真值的一小部分。它不是 $F_j$ 的估计量，不进图也不进表。

**采样率**。全部取 $p \le 0.05$，只有 5%–19% 的真值类型留下观测。这比论文真实列实验的 $p \ge 0.1$ 更稀疏，属于边界情形而非论文报告的池化精度。

**必须先说清的口径问题**。上一版这张图只画了 oracle 先验下的解析逆。`prior_from_truth` 用完整总体的真实频次构造先验，等于把答案放进先验，所以哪怕 $p=0.01$，曲线也几乎与真值完全重合。那是后验公式的自洽性检验，不是从样本复原。真值当时又是一条画在粗红线下面的细黑线，实际上看不见。采样率这么低时，任何只看样本的方法都不该复原得如此精确。

这一版改了两处。真值最后绘制，用粗黑线加空心黑点压在所有估计曲线之上。解析逆分成两条口径。**部署态**用 `fitted_prior_eb` 只从稀疏样本拟合先验。**信息上限**用 `prior_from_truth` 从完整总体构造先验，作为灰虚线单独放在下面的偏离图里，明确标注不是复原结果。

**图输出**。脚本同时输出 PDF、EPS、SVG 三种矢量格式，PNG 只作预览。


In [ ]:
from d4d_baselines import (spectrum_naive, spectrum_good_turing,
                           spectrum_chebyshev, spectrum_ours,
                           distribution_metrics, H_CAP)
from d4d_metrics import prior_from_truth
from d4d_modern_baselines import smooth_bayes_params, spectrum_smooth_bayes
from eb_prior import fitted_prior_eb
from matplotlib.lines import Line2D

CASES = [
    dict(alpha=1.5, n=1_000_000, vocab=100_000, p=0.01, seed=11, label='Zipf a1.5, N1.0M, p0.01'),
    dict(alpha=1.5, n=300_000,  vocab=60_000,  p=0.02, seed=3,  label='Zipf a1.5, N300k, p0.02'),
    dict(alpha=1.3, n=100_000,  vocab=30_000,  p=0.02, seed=0,  label='Zipf a1.3, N100k, p0.02'),
    dict(alpha=1.8, n=100_000,  vocab=30_000,  p=0.05, seed=0,  label='Zipf a1.8, N100k, p0.05'),
]
FIT = 'Ours (fitted prior)'
ORC = 'Ours (oracle prior)'
GT  = 'Good-Turing'
NV  = 'Naive 1/p'
CH  = 'Chebyshev'
SB  = 'SB linear (fitted)'
METHODS = [NV, GT, CH, SB, FIT, ORC]

def fof_case(cfg):
    r2 = np.random.default_rng(cfg['seed'])
    counts = zipf_population(r2, cfg['n'], cfg['vocab'], cfg['alpha'])
    f_true = counts[counts > 0].astype(np.int64)
    g = thin(f_true, cfg['p'], r2)
    f_samp = fofo_spectrum(g)                      # 经典基线输入，不是对比曲线
    F_true = fofo_spectrum(f_true, j_max=H_CAP)    # 真值，在 f 轴上
    d_obs = int((g >= 1).sum()); N_total = int(f_true.sum())

    # 部署态先验，只从稀疏样本拟合，不接触完整总体
    N_max = max(int(g.sum() / cfg['p']) + 64, d_obs + 64)
    pf_fit, _, a0_eb, b0_eb = fitted_prior_eb(g, cfg['p'], N_max)
    lf_fit = gammaln(np.arange(N_max + 513) + 1.0)

    # 信息上限先验，直接由完整总体构造，只作对照
    pf_orc, lf_orc = prior_from_truth(f_true, N_total)
    a0_sb, b0_sb = smooth_bayes_params(f_samp, cfg['p'], int(g.sum()))

    sp = {
        NV:  spectrum_naive(g, cfg['p']),
        GT:  spectrum_good_turing(f_samp),
        CH:  spectrum_chebyshev(f_samp),
        SB:  spectrum_smooth_bayes(g, cfg['p'], a0_sb, b0_sb, d_obs),
        FIT: spectrum_ours(g, cfg['p'], pf_fit, lf_fit, N_max, d_obs),
        ORC: spectrum_ours(g, cfg['p'], pf_orc, lf_orc, N_total, d_obs),
    }
    return F_true, sp, d_obs / max(F_true.sum(), 1.0)

rows = [fof_case(cfg) for cfg in CASES]
J = 32
STY = {
    FIT: dict(color='#D55E00', lw=2.6, marker='o', ms=3.0, zorder=12),
    ORC: dict(color='#8C8C8C', lw=1.7, ls='--', zorder=8),
    GT:  dict(color='#0072B2', lw=1.7, marker='s', ms=2.4, zorder=5),
    NV:  dict(color='#CC79A7', lw=1.7, ls='--', marker='^', ms=3.2, zorder=4),
    CH:  dict(color='#009E73', lw=1.7, ls='-.', zorder=4),
    SB:  dict(color='#6A3D9A', lw=1.7, ls=':', marker='D', ms=3.0, zorder=4),
}
LBL = {FIT: 'Ours, prior fitted from sample', ORC: 'Ours, oracle prior (ceiling)',
       GT: GT, NV: NV, CH: CH, SB: SB}
DRAW = [GT, NV, CH, SB, FIT]   # oracle 上限压在真值线上，放到偏离图里画

def cv(a):
    v = np.asarray(a, float)[:J].copy()
    v[v <= 0] = np.nan        # 真零不画成假的下限
    return v

fig, axes = plt.subplots(2, 2, figsize=(11, 7.4), sharex=True)
x = np.arange(1, J + 1)
for ax, cfg, (F_true, sp, frac) in zip(axes.ravel(), CASES, rows):
    for k in DRAW:
        ax.plot(x, cv(sp[k]), label=LBL[k], **STY[k])
    # 真值最后画，粗黑线加空心黑点，压在所有估计曲线之上
    ax.plot(x, cv(F_true), color='#000000', lw=2.4,
            label='Ground truth F_j (full population)', zorder=30)
    ax.plot(x, cv(F_true), 'o', ms=5.6, mfc='white', mec='#000000',
            mew=1.5, zorder=31)
    ax.set_yscale('log'); ax.set_ylim(bottom=0.5); ax.set_xlim(1, J)
    ax.set_xticks([1, 5, 10, 15, 20, 25, 32])
    ax.set_title(cfg['label'], fontsize=9.5, fontweight='bold')
    tvf = distribution_metrics(F_true, sp[FIT])['tv']
    tvo = distribution_metrics(F_true, sp[ORC])['tv']
    ax.text(0.97, 0.95,
            f'observed types {frac:.1%} of truth\nfitted TV {tvf:.3f}  |  oracle TV {tvo:.3f}'
            '\noracle ceiling sits under the truth line',
            transform=ax.transAxes, ha='right', va='top', fontsize=7.7,
            color='#222222', linespacing=1.35,
            bbox=dict(fc='white', ec='#BBBBBB', lw=0.6, pad=2.0))
for ax in axes[1]:
    ax.set_xlabel('true frequency j in the full population')
for ax in axes[:, 0]:
    ax.set_ylabel(r'FoF  $F_j$  (number of types)')
handles, labels = axes[0, 0].get_legend_handles_labels()
handles, labels = list(handles), list(labels)
k = labels.index('Ground truth F_j (full population)')
handles[k] = Line2D([], [], color='#000000', lw=2.4, marker='o', ms=5.6,
                    mfc='white', mec='#000000', mew=1.5)
order = [k] + [labels.index(LBL[m]) for m in [FIT, GT, NV, CH, SB]]
fig.legend([handles[i] for i in order], [labels[i] for i in order],
           loc='upper center', ncol=3, frameon=False,
           bbox_to_anchor=(0.5, 0.955), fontsize=8.6,
           columnspacing=1.6, handlelength=2.6)
fig.suptitle('FoF recovery at low retention against the full-population ground truth',
             y=0.987, fontsize=11.5, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, 0.865))
plt.show()


**怎么读这张图**。粗黑线加空心黑点是完整总体的 FoF 真值，现在它是最后绘制的图层，不会被任何估计曲线盖住。朱红粗线是解析逆在**只从样本拟合先验**下的后验均值 FoF。它在小 $j$ 处明显低于真值，在尾部又高于真值，TV 在 0.37–0.44。这才是 $p \le 0.05$ 时真实的复原水平。右上角同时给出 oracle 上限的 TV，约 0.006–0.026，两者差距就是先验泄漏带来的虚假精度。

其余方法都在同一个 $f$ 轴上估计 $F_j$。Good–Turing 把 $F_1$ 低估近一个数量级并在高 $j$ 处产生孤立尖峰，Chebyshev 几乎给出一条水平线，朴素 $1/p$ 把观测计数搬到 $j \approx 1/p$（低采样率下超出显示范围），SB linear 用拟合先验时几乎把质量全压进首桶。这些基线没有显式建模二项稀疏信道，在如此低的保留率下失配明显。

**注意**。这四个合成案例比论文真实列实验稀疏得多。论文的池化精度在 $p \ge 0.1$ 上报告，这里做的是边界研究，不能拿来代表论文表格里的数字。


In [ ]:
print(f"{'case':<28} {'method':<24} {'W1':>7} {'TV':>8} {'RMSE':>9} {'NDV rel':>9}")
print('-' * 90)
for cfg, (F_true, sp, frac) in zip(CASES, rows):
    for name in METHODS:
        F = sp[name]
        m = distribution_metrics(F_true, F)
        ndv = (F.sum() - F_true.sum()) / F_true.sum()
        print(f"{cfg['label']:<28} {name:<24} {m['w1']:>7.2f} {m['tv']:>8.4f} "
              f"{m['rmse']:>9.5f} {ndv:>+8.1%}")
    print()

# 偏离图。(a) 每个 j 上的估计/真值比，1 表示完全复原。
# (b) 固定一个总体扫描 p，看采样率变低时复原误差怎么变。
J_RATIO = 16
COLORS = ['#D55E00', '#0072B2', '#009E73', '#6A3D9A']
SWEEP_P = [0.002, 0.005, 0.01, 0.02, 0.05, 0.1]
r0 = np.random.default_rng(3)
cnt = zipf_population(r0, 300_000, 60_000, 1.5)
f0 = cnt[cnt > 0].astype(np.int64)
Ft0 = fofo_spectrum(f0, j_max=H_CAP)
pf0, lf0 = prior_from_truth(f0, int(f0.sum()))
sweep = []
for i, pv in enumerate(SWEEP_P):
    rr = np.random.default_rng(3000 + i)
    gg = thin(f0, pv, rr)
    dd = int((gg >= 1).sum())
    fs0 = fofo_spectrum(gg)
    Nm = max(int(gg.sum() / pv) + 64, dd + 64)
    pf, _, _, _ = fitted_prior_eb(gg, pv, Nm)
    lf = gammaln(np.arange(Nm + 513) + 1.0)
    est_f = spectrum_ours(gg, pv, pf, lf, Nm, dd)
    est_o = spectrum_ours(gg, pv, pf0, lf0, int(f0.sum()), dd)
    est_g = spectrum_good_turing(fs0)
    sweep.append((pv, dd / Ft0.sum(),
                  distribution_metrics(Ft0, est_f)['tv'],
                  distribution_metrics(Ft0, est_o)['tv'],
                  distribution_metrics(Ft0, est_g)['tv']))

fig2, axs = plt.subplots(1, 2, figsize=(11.6, 4.1))
ax = axs[0]
xr = np.arange(1, J_RATIO + 1)
for idx, (cfg, (F_true, sp, frac)) in enumerate(zip(CASES, rows)):
    t = np.asarray(F_true, float)[:J_RATIO]
    ok = t > 0
    ratio = np.full(J_RATIO, np.nan)
    ratio[ok] = np.asarray(sp[FIT], float)[:J_RATIO][ok] / t[ok]
    ro = np.full(J_RATIO, np.nan)
    ro[ok] = np.asarray(sp[ORC], float)[:J_RATIO][ok] / t[ok]
    ax.plot(xr, ratio, color=COLORS[idx], lw=1.9, label=cfg['label'])
    ax.plot(xr, ro, color='#8C8C8C', lw=1.1, ls='--')
ax.axhline(1.0, color='#000000', lw=1.6, ls=':')
ax.set_ylim(0, 2.8); ax.set_xlim(1, J_RATIO)
ax.set_xlabel('true frequency j in the full population')
ax.set_ylabel(r'ratio  $\hat F_j / F_j$')
ax.set_title('(a) distance from truth, one at perfect recovery', fontsize=9.5)
ax.text(0.02, 0.04, 'solid coloured, fitted prior\ngrey dashed, oracle ceiling\n'
        'j capped at 16, larger j has small F_j',
        transform=ax.transAxes, fontsize=7.6,
        bbox=dict(fc='white', ec='#BBBBBB', lw=0.6, pad=2.0))
ax.legend(loc='upper right', fontsize=7.0, frameon=True, handlelength=1.8)

ax = axs[1]
pvals = [s[0] for s in sweep]
ax.plot(pvals, [s[2] for s in sweep], color='#D55E00', lw=2.4, marker='o',
        ms=4.5, label='Ours, fitted prior')
ax.plot(pvals, [s[3] for s in sweep], color='#8C8C8C', lw=1.8, ls='--',
        marker='s', ms=4.0, label='Ours, oracle ceiling')
ax.plot(pvals, [s[4] for s in sweep], color='#0072B2', lw=1.8, ls='-.',
        marker='^', ms=4.0, label='Good-Turing')
ax.set_xscale('log'); ax.set_ylim(0, 0.58)
ax.set_xlabel('retention rate p of the thinning channel')
ax.set_ylabel('total variation of the normalized FoF')
ax.set_title('(b) recovery degrades as the channel gets sparser', fontsize=9.5)
ax.legend(loc='upper right', fontsize=7.6)
ax2 = ax.twinx()
ax2.plot(pvals, [s[1] for s in sweep], color='#BBBBBB', lw=1.2, ls=':',
         marker='D', ms=3.2)
ax2.set_ylim(0, 0.32)
ax2.set_ylabel('observed types as a fraction of truth', color='#777777', fontsize=8)
ax2.grid(False)
fig2.suptitle('Why a near-perfect overlap at low retention is not a recovery result',
              y=0.99, fontsize=11, fontweight='bold')
fig2.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

print(f"\n{'p':<8} {'observed types':>15} {'TV fitted':>11} {'TV oracle':>11} {'TV GT':>9}")
for s in sweep:
    print(f"{s[0]:<8.3f} {s[1]:>14.1%} {s[2]:>11.4f} {s[3]:>11.4f} {s[4]:>9.4f}")


W1、TV、RMSE 在截断并归一化的频谱上计算，NDV 相对误差在**绝对数量**上计算，表中为有符号的 $(\hat D - D)/D$。

**诚实边界**。部署态（fitted prior）在这些案例里 TV 为 0.37–0.44，NDV 低估 47%–71%。`fitted_prior_eb` 的 Gamma–Poisson 形状参数在四个案例里都顶到了下界，因为观测频谱几乎没有 doubleton，样本无法识别先验形状。这是低采样率下真正的瓶颈。oracle 上限的 TV 只有 0.006–0.026，但它的先验来自完整总体，属于信息上限，不是复原结果。

偏离图的 (b) 直接量化了用户关心的那件事。固定一个总体，$p$ 从 0.1 降到 0.002，有观测的类型占比从 24.6% 降到 1.7%，部署态 TV 从 0.29 升到 0.51，而 oracle 上限始终在 0.01 附近。**采样率越低，只看样本的方法越难贴近真值**。采样率低还能完全重合，只可能是先验里泄进了答案。

论文在 $p \ge 0.1$ 的真实列上报告池化精度，这里的合成低采样率是边界研究。SB linear 用的是自己拟合的 Gamma–Poisson 先验，与部署态并非严格同先验对照，论文的 `tab_factor_control` 才做了同先验因子对照。

**图格式**。每张图都有 PDF、EPS、SVG 三种矢量版本，PNG 只作预览。完整数字与两张图的矢量文件在 `make_fof_recovery_figure.py`、`figures/fof_recovery_comparison.*`、`figures/fof_recovery_deviation.*` 与 `experiments/results/fof_recovery_figure/`。


## 5. 90% plausible set（校准的不确定性带）

只给点估计不够。论文用后验**抽样** $K$ 个完整总体向量，每个重新分桶成频谱，再取每列 5%/95% 分位，得到 90% 频段。它不需要真值来构造（模型内后验），而 empirical coverage（真值落在带内的比例）需要真值来**评估**——这两者必须分开。

下面在单个 Zipf 总体上跑 `plausible_set`，画频段 + 真值 + 观测 + 后验均值，并打印逐类型 95% 覆盖率和校准曲线。

In [ ]:
K = 400
res = plausible_set(c_true, c_obs, p, K, prior, logfact, N, rng)
J = 50
fig, ax = plt.subplots()
js = np.arange(1, J + 1)
ax.fill_between(js, res['band_lo'][:J], res['band_hi'][:J], alpha=0.25, color='C0', label='90% band')
ax.plot(res['mean_spec'][:J], 'C0', label='posterior mean')
ax.plot(res['truth_spec'][:J], 'k-', lw=1.5, label='truth (observed types)')
ax.plot(res['obs_spec'][:J], 'gray', label='observed')
ax.set_yscale('log'); ax.set_xlabel('j'); ax.set_ylabel('F_j (log)')
ax.set_title('90% plausible set'); ax.legend(loc='lower left'); plt.tight_layout(); plt.show()

print('per-type 95% coverage =', round(res['coverage95'], 4))
cal = res['cov95_line']
print('nominal -> empirical coverage:')
for nom, emp in cal.items():
    print(f'  {nom:.2f} -> {emp:.3f}')

如果 empirical 接近 nominal（例如 0.90 名义 → ~0.90 经验），说明频段是**校准**的。论文强调：后验内容（构造带）和 empirical coverage（评估带）是两个不同概念，不能混为一谈；per-type、per-frequency、simultaneous 覆盖也必须分开报告。

## 6. 学习组件（Thickener / endpoint regression）的定位与诚实局限

到这一步，所有统计信息都来自**解析逆**。那为什么要训练网络？

当先验**没有闭式**（例如真实分布不是 Gamma–Poisson，或要处理联合分布），后验均值无法手算，论文用 `Thickener` 去**学** endpoint 回归目标：

$$R_s(c) = \mathbb{E}[c_{\text{target}} \mid c_s = c],$$

即用 set-transformer 在分桶频谱上回归"下一个更密层级"的频谱。在 Gamma–Poisson 共轭信道上，这个 endpoint **恰好等于第 2 节那条仿射闭式**——所以学到的网络只是去逼近它。下面画出闭式 endpoint $R(g)=\mathbb{E}[c\mid g]$ 随 $g$ 的曲线，直观说明"在共轭信道上，学习组件学的是一个简单仿射映射"。

In [ ]:
gs = np.arange(1, 201)
Rs = np.array([(g + (1 - p) * (a0 + g) / (b0 + p)) for g in gs])
fig, ax = plt.subplots()
ax.plot(gs, Rs, 'C0', lw=2, label=r'closed-form endpoint $R(g)=\mathbb{E}[c|g]$')
ax.plot(gs, gs / p, 'k--', lw=1, label=r'naive $g/p$')
ax.set_xlabel('observed g'); ax.set_ylabel('endpoint (predicted c)')
ax.set_title('endpoint regression target on the conjugate channel')
ax.legend(); plt.tight_layout(); plt.show()

**诚实局限（论文 Table II / Algorithm 2 讨论）**：学习组件有 1200 步的训练预算上限，800→1600 步 W1 从 4.942 到 4.950（k=5）不再提升；zero-shot 配置在所有设置下都不如原始样本。因此**论文贡献落在 Algorithm 1 的解析逆**，学习组件只为处理非共轭先验而存在，不应被当成主要卖点。

完整训练入口（需要 torch）：`train_thickener(cfg)`、`Thickener`、`chain_thickener(model, ...)`，默认超参见 `TrainConfig`（AdamW `lr=3e-4`、wd `0.01`、`huber_beta=1.0`、梯度裁剪 `5.0`）。大规模训练与消融脚本在 `experiments_archive/`。

## 7. 三条诚实的方法边界

论文明确界定了方法的适用范围，不夸大：

1. **重尾精度仍归于原始样本**：解析逆对长尾不 uniformly 占优，原始样本在尾部仍是强基线。
2. **解析逆不 uniform 支配最强线性估计器**：在 self-join 等下游泛函上，解析逆只赢配对显著的子集（对 Good–Turing 仅 ~50%，对 SB linear 仅 ~35%）。
3. **单列先验不迁移到联合分布**：把单列拟合先验直接用到 joint cell 上，会差 1–2 个数量级（occupied joint cell 平均 1.33–11.39 元组 vs marginal 200–500），联合分布恢复在实验中是 prior-transfer 失效，如实写成限制而非胜利。

方法的适用范围是：**单列、单一已知 Bernoulli 信道下的后验恢复**。

## 8. 代码 ↔ 论文 对照表

| `d4d_logic.py` 符号 | 论文对应 | 说明 |
|---|---|---|
| `thin(c, p, rng)` | 前向加噪 / Algorithm 1 的 noising | 二项稀疏，每条记录独立以 $p$ 保留 |
| `analytic_posterior(g, p, prior, logfact, N)` | Algorithm 1 解析逆 | 通用离散后验 $P(c\mid g)\propto P(c)\mathrm{Bin}(g\mid c,p)$ |
| `gamma_poisson_prior` / 闭式 $\mathbb{E}[c\mid g]$ | Section V 共轭闭式 | Gamma–Poisson 下退化为仿射均值 |
| `posterior_mean_per_type`（本 notebook） | 点估计 | 逐类型后验均值，用于逐类型 MAE 对比 |
| `spectrum_ours`（`d4d_baselines`） | Table IV / 第 4 节 FoF 图 | 后验质量按 $P(f\mid g)$ 摊入频谱 + 未见类型补偿 |
| `plausible_set(...)` | 90% plausible set | 后验抽样 → 频段，模型内构造 |
| `Thickener` / `train_thickener` / `chain_thickener` | Algorithm 2 学习端点 | 非共轭先验下的 endpoint 回归，有预算上限 |
| `w1_distance` / `fofo_spectrum` | 指标 / 频谱 | 截断 renormalized W1；频率频谱 $F_j$ |

**复现入口**（active 实验脚本，均在 `experiments/code/`）：
- `analyze_prior_and_joint.py` —— 生成正文 Table III（先验敏感性）与 Table VII（联合尺度诊断）
- `validate_downstream_fair_comparison.py` —— 下游公平比较与配对检验
- `validate_*.py` 系列 —— 各类基线 / 真实数据 / joint 验证
- `make_hist_figures.py` / `make_publication_figures.py` —— 论文图
- `experiments_archive/` —— 一次性消融实验存档（不在活跃构建中）

**下一步**：如果你只想理解主算法，第 1–3 节覆盖了解析逆与逐类型恢复，第 4 节用 FoF 折线图直观对比了复原效果，第 5 节给出校准的不确定性带；第 6 节解释了为什么还需要一个学习组件、以及它的边界。